In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path().resolve()

In [ ]:
ground_truth = pd.read_csv(ROOT / 'ground_truth' / 'landmark_tracks' / 'landmark_tracks.csv')
prediction_target = pd.read_csv(ROOT / 'prediction_data' / 'prediction_landmark_tracks' / 'landmark_tracks.csv')

In [ ]:
frame_start = 60
frame_end = 145
# the movment of the landmark in the ground truth data between frame 60 and 145
ground_truth_movement = ground_truth[ground_truth['frame'].between(frame_start, frame_end)]

In [ ]:
ground_truth_movement

In [ ]:
prediction_target.frame.min(), prediction_target.frame.max()

In [ ]:
# split the prediction target into movement and check window with overlap=90 frames

for i in range(prediction_target.frame.min(), prediction_target.frame.max(), 10):
    start = i
    end = i + 90
    check_window = prediction_target.iloc[start:end]
    print(f"frame range: {start}-{end}, check_window shape: {check_window.shape}")


Create the matrices we want to compare

In [ ]:
import numpy as np

landmarks = ground_truth_movement['landmark'].unique()
frames = sorted(ground_truth_movement['frame'].unique())

# pivot: rows=frame, columns=landmark, values=[x, y]
pivot_x = ground_truth_movement.pivot(index='frame', columns='landmark', values='x').reindex(columns=landmarks)
pivot_y = ground_truth_movement.pivot(index='frame', columns='landmark', values='y').reindex(columns=landmarks)

# stack into (num_frames, num_landmarks, 2) matrix — channel per landmark
ground_truth_matrix = np.stack([pivot_x.values, pivot_y.values], axis=-1)
print(f"shape: {ground_truth_matrix.shape}  # (frames, landmarks, xy)")
print(f"landmarks: {list(landmarks)}")

In [ ]:
ground_truth_matrix

In [ ]:
ground_truth_matrix.shape

extract window matrices

In [ ]:
window_size = 90
step = 10

prediction_windows = []

for i in range(prediction_target.frame.min(), prediction_target.frame.max(), step):
    start = i
    end = i + window_size
    window_df = prediction_target[prediction_target['frame'].between(start, end - 1)]
    
    if window_df.empty:
        continue
    
    pivot_x = window_df.pivot(index='frame', columns='landmark', values='x').reindex(columns=landmarks)
    pivot_y = window_df.pivot(index='frame', columns='landmark', values='y').reindex(columns=landmarks)
    
    matrix = np.stack([pivot_x.values, pivot_y.values], axis=-1)
    prediction_windows.append({'start': start, 'end': end, 'matrix': matrix})
    
print(f"total windows: {len(prediction_windows)}")
for w in prediction_windows:
    print(f"  frames {w['start']}-{w['end']}, shape: {w['matrix'].shape}")

In [ ]:
def normalize_pose_matrix(matrix):
    """Center on mean position per frame, scale to [0,1] range."""
    # center: subtract mean landmark position per frame → removes translation
    center = matrix.mean(axis=1, keepdims=True)  # (frames, 1, 2)
    centered = matrix - center
    # scale: divide by max absolute value → removes scale differences
    scale = np.abs(centered).max()
    if scale > 0:
        centered = centered / scale
    return centered

# normalize ground truth
ground_truth_norm = normalize_pose_matrix(ground_truth_matrix)

# normalize each prediction window
prediction_windows_norm = []
for w in prediction_windows:
    norm_matrix = normalize_pose_matrix(w['matrix'])
    prediction_windows_norm.append({'start': w['start'], 'end': w['end'], 'matrix': norm_matrix})

print(f"ground_truth_norm shape: {ground_truth_norm.shape}, range: [{ground_truth_norm.min():.3f}, {ground_truth_norm.max():.3f}]")
print(f"example window norm shape: {prediction_windows_norm[0]['matrix'].shape}, range: [{prediction_windows_norm[0]['matrix'].min():.3f}, {prediction_windows_norm[0]['matrix'].max():.3f}]")

compare frames

In [ ]:
# truncate to min frame count and compute MSE for each window
gt = ground_truth_norm
n_gt_frames = gt.shape[0]

mse_results = []
for w in prediction_windows_norm:
    pred = w['matrix']
    n_frames = min(n_gt_frames, pred.shape[0])
    
    # truncate both to the shorter length
    gt_trunc = gt[:n_frames]
    pred_trunc = pred[:n_frames]
    
    mse = np.mean((gt_trunc - pred_trunc) ** 2)
    mse_results.append({'start': w['start'], 'end': w['end'], 'mse': mse})

mse_df = pd.DataFrame(mse_results).sort_values('mse')
print(mse_df.to_string(index=False))
print(f"\nbest match: frames {mse_df.iloc[0]['start']:.0f}-{mse_df.iloc[0]['end']:.0f}, MSE={mse_df.iloc[0]['mse']:.6f}")

In [ ]:
import matplotlib.pyplot as plt

mse_sorted = mse_df.sort_values('start')
threshold = mse_sorted['mse'].quantile(0.2)

# first window below threshold → region start; best MSE window → annotation point
region_start = mse_sorted[mse_sorted['mse'] < threshold]['start'].min()
best = mse_df.sort_values('mse').iloc[0]
region_end = best['end']

fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.RdYlGn_r((mse_sorted['mse'] - mse_sorted['mse'].min()) / (mse_sorted['mse'].max() - mse_sorted['mse'].min()))
ax.bar(mse_sorted['start'], mse_sorted['mse'], width=8, color=colors, alpha=0.8, label='MSE per window')

# highlight region from first-below-threshold to end of best window
ax.axvspan(region_start, region_end, alpha=0.2, color='green', label=f"Detected region: {region_start:.0f}-{region_end:.0f}")
ax.scatter(best['start'], best['mse'], color='green', s=150, zorder=5, edgecolors='black', linewidth=2)
ax.annotate(f"MSE={best['mse']:.4f}\nframes {region_start:.0f}-{region_end:.0f}",
            xy=(best['start'], best['mse']),
            xytext=(best['start'] + 30, best['mse'] + 0.02),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=10, color='green')

ax.axhline(y=threshold, color='orange', linestyle='--', alpha=0.7, label=f'Top 20% threshold ({threshold:.4f})')

ax.set_xlabel('Window Start Frame', fontsize=12)
ax.set_ylabel('MSE', fontsize=12)
ax.set_title('MSE: Prediction Windows vs Ground Truth (Biellman)', fontsize=14)
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.set_xticks(mse_sorted['start'].values[::2])
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


start looks like 180 , end is 290

In [ ]:
import cv2
import os
import glob

frames_dir = str(ROOT / 'prediction_data' / 'prediction_frames')
output_path = str(ROOT / 'biellman_detected_result.mp4')

best_start = 180
best_end = 290
best_mse = mse_df[mse_df['start'] == best_start]['mse'].values[0]

# get all frames sorted
all_frames = sorted(glob.glob(os.path.join(frames_dir, 'frame_*.jpg')))
total_frames = len(all_frames)

# read first frame to get dimensions
sample = cv2.imread(all_frames[0])
h, w = sample.shape[:2]

# ensure even dimensions (required by most video codecs)
w_out = w - (w % 2)
h_out = h - (h % 2)

fourcc = cv2.VideoWriter_fourcc(*'avc1')
fps = 30
out = cv2.VideoWriter(output_path, fourcc, fps, (w_out, h_out))

if not out.isOpened():
    # fallback to mp4v if avc1 not available
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (w_out, h_out))

for frame_path in all_frames:
    frame_idx = int(os.path.basename(frame_path).split('_')[1].split('.')[0])
    frame = cv2.imread(frame_path)
    
    if frame is None:
        continue
    
    # ensure frame matches expected output size
    fh, fw = frame.shape[:2]
    if fw != w_out or fh != h_out:
        frame = cv2.resize(frame, (w_out, h_out))
    
    is_detected = best_start <= frame_idx < best_end
    
    if is_detected:
        # green overlay bar at top
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w_out, 70), (0, 100, 0), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
        
        label = f"Biellman Detected | MSE={best_mse:.4f}"
        cv2.putText(frame, label, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        
        frame_label = f"Frame {frame_idx} / {best_start}-{best_end}"
        cv2.putText(frame, frame_label, (15, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1, cv2.LINE_AA)
    else:
        # subtle frame counter on non-detected frames
        cv2.putText(frame, f"Frame {frame_idx}", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 1, cv2.LINE_AA)
    
    out.write(frame)

out.release()
print(f"Video saved to {output_path}")
print(f"Total frames: {total_frames}, Detection: {best_start}-{best_end}, Resolution: {w_out}x{h_out}, FPS: {fps}")